In [1]:
# ============================================================================
# notebook: notebooks/01_cohort_and_model.ipynb  (clean rebuild, new paths)
# Stage 1: freeze the analysis cohort + audited model + group tags.
# Reads results/stage0_labeled.parquet. Writes results/. Run from notebooks/.
# ============================================================================


# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Paths, imports, load Stage-0 output
# ─────────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold

ROOT    = Path("..").resolve()
RESULTS = ROOT / "results"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

AUDIT_AXIS = ["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1,7)] + [f"PAY_AMT{i}" for i in range(1,7)]
BORDERLINE_BAND = (0.40, 0.60)
MIN_CELL_N = 30

df = pd.read_parquet(RESULTS / "stage0_labeled.parquet")
print("Loaded stage0_labeled:", df.shape)


# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Cell key + group tags (disadvantaged / advantaged, empirical)
# ─────────────────────────────────────────────────────────────────────────
df["CELL"] = df["SEX_LBL"].astype(str)+"·"+df["AGE_BAND"].astype(str)+"·"+df["EDU_BAND"].astype(str)

PRIMARY_DIS   = ["M·20s·univ","F·20s·univ","M·30s·univ"]
SECONDARY_DIS = ["M·40s·univ","M·40s·hs","F·40s·hs"]

cell_rate = (df.groupby("CELL")
               .agg(approval_rate=("APPROVED","mean"),
                    borderline_appr=("VIP_BORDERLINE",
                        lambda s:int(((s==1)&(df.loc[s.index,"APPROVED"]==1)).sum())))
               .reset_index())
adv_pool = cell_rate[cell_rate["borderline_appr"]>=50].sort_values("approval_rate",ascending=False)
ADVANTAGED = adv_pool.head(3)["CELL"].tolist()

def tag(c):
    if c in PRIMARY_DIS:   return "dis_primary"
    if c in SECONDARY_DIS: return "dis_secondary"
    if c in ADVANTAGED:    return "advantaged"
    if str(c).startswith("F·60+"): return "highlight_F60"
    return "other"
df["GROUP"] = df["CELL"].apply(tag)
print("Advantaged cells:", ADVANTAGED)
print(df.loc[df.APPROVED==1,"GROUP"].value_counts().to_dict())


# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Audited model (audit axis only), reproduce OOF prob, freeze cohorts
# ─────────────────────────────────────────────────────────────────────────
scaler = StandardScaler().fit(df[AUDIT_AXIS].values)
X_all = scaler.transform(df[AUDIT_AXIS].values)
y_all = df["VIP_CLEAR"].values

cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                            random_state=RANDOM_STATE, n_jobs=-1)
df["P_VIP_stage1"] = cross_val_predict(rf, X_all, y_all, cv=cv,
                                       method="predict_proba", n_jobs=-1)[:,1]
lo, hi = BORDERLINE_BAND
df["VIP_BORDERLINE_s1"] = ((df["P_VIP_stage1"]>=lo)&(df["P_VIP_stage1"]<=hi)).astype(int)

rf_final = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                  random_state=RANDOM_STATE, n_jobs=-1).fit(X_all, y_all)

approved_idx   = df.index[df["APPROVED"]==1].to_numpy()
borderline_idx = df.index[(df["APPROVED"]==1)&(df["VIP_BORDERLINE_s1"]==1)].to_numpy()
print(f"Approved: {len(approved_idx):,}  Borderline: {len(borderline_idx):,}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Persist Stage-1 artifacts
# ─────────────────────────────────────────────────────────────────────────
np.save(RESULTS/"stage1_X_audit_scaled.npy", X_all)
np.save(RESULTS/"stage1_approved_idx.npy", approved_idx)
np.save(RESULTS/"stage1_borderline_idx.npy", borderline_idx)
joblib.dump(scaler,   RESULTS/"stage1_scaler.joblib")
joblib.dump(rf_final, RESULTS/"stage1_rf_final.joblib")
keep = AUDIT_AXIS + ["VIP_CLEAR","APPROVED","P_VIP_stage1","VIP_BORDERLINE_s1",
                     "SEX","AGE","EDUCATION","SEX_LBL","AGE_BAND","EDU_BAND","CELL","GROUP"]
df[keep].to_parquet(RESULTS/"stage1_cohort.parquet")
print("Saved Stage-1 artifacts to results/.")
print("=== STAGE 1 COMPLETE ===")

Loaded stage0_labeled: (30000, 34)
Advantaged cells: ['F·30s·grad', 'M·30s·grad', 'M·20s·grad']
{'other': 5437, 'advantaged': 2641, 'dis_primary': 2067, 'dis_secondary': 903, 'highlight_F60': 41}
Approved: 11,089  Borderline: 1,141
Saved Stage-1 artifacts to results/.
=== STAGE 1 COMPLETE ===
